# Day 192 — Final 12-Month Capstone: Kickoff (Project Selection + Architecture Plan)

**Month 12 · Day 1 of Final Capstone (Days 192–196)**

Month 11 closed at **1010/1010 + 130★, all 11 days perfect.** This is the last capstone of the
12-month program — it needs to demonstrate everything from Months 6–11 working together as one
system, not five separate lesson exercises:

| Month | Skill it must show up as |
|---|---|
| 6–7 | Classical ML — a real predictive model (classification/regression) |
| 8 | Deployment — Streamlit or FastAPI serving the model |
| 9 | NLP / Deep Learning — text feature extraction or a transformer step |
| 10 | LangChain + RAG — an LLM layer answering questions over the data |
| 11 | Data Engineering — a pipeline class, tests, and (optionally) CI/CD |

Today is **not** a coding-heavy day. It's the day the architecture gets decided on paper before
a single production line of code is written — same discipline as Day 189 (Month 11 Capstone Kickoff),
one level up in scope.


## Concept Notes — Why Capstone Architecture Comes Before Capstone Code

**The mistake this day exists to prevent:** jumping straight into `pip install` and notebook cells
for a project this size. A 5-layer integrated system without an architecture decision made first
tends to end up as five disconnected notebooks bolted together in Week 5, which is exactly what a
client-facing final capstone should never look like.

**Real-world use:** this is what a solutions architect or senior data scientist does before scoping
a client SOW — pick the dataset, define the layer boundaries, decide what tech sits in each layer,
and write down what "done" means *before* estimating time or writing code. It is the same reason
Day 189 opened Month 11's capstone with an architecture-first task rather than a code-first one.

**The five architecture layers this capstone will have** (you'll define these in Task 2):
1. **Ingestion** — how raw data enters the system
2. **Transformation / Cleaning** — where raw data becomes trustworthy data
3. **ML / Feature Layer** — where a trained model produces a prediction
4. **RAG / LLM Layer** — where an LLM answers natural-language questions over the data
5. **Serving / Monitoring** — how a user or client reaches the system, and how you'd know if it broke


---
## Section 1 — Raw Data (DO NOT MODIFY)

**Dataset: TeleServe India** — a synthetic telecom customer-service dataset (seed=192) chosen
specifically because it has *both* a structured half (billing/usage → classic ML) and a text half
(complaint transcripts → NLP/RAG), which is exactly the shape the final capstone architecture needs
to justify combining an ML layer with an LLM layer in the same pipeline.

This cell is locked for the rest of the capstone arc (Days 192–196). Every later day re-reads this
same generator — never a re-randomized version — so results stay reproducible end to end.


In [1]:
# ============================================================
# RAW DATA GENERATOR — TeleServe India (seed=192) — DO NOT MODIFY
# Locked for the entire Month 12 Final Capstone arc (Days 192-196)
# ============================================================
import pandas as pd
import numpy as np
import random

SEED = 192
random.seed(SEED)
np.random.seed(SEED)

N = 400

plan_types_clean = ["Prepaid", "Postpaid", "Broadband"]
# a few rows get inconsistent casing on purpose - a Day 193 data-layer cleaning target
plan_types_dirty = ["prepaid", "POSTPAID", "Broadband "]

complaint_templates = [
    "Network signal keeps dropping in {area} area since last week",
    "Billed amount does not match my selected {plan} plan this month",
    "Internet speed far below promised {plan} plan speed during evening hours",
    "Recharge done but plan benefits not activated for 2 days",
    "Customer care call disconnected three times without resolving {issue}",
    "Data balance deducted incorrectly after using app for {issue}",
    "Broadband router keeps restarting every few hours in {area}",
    "Requested plan downgrade last month but still billed for old plan",
    "SMS service not working since the last recharge cycle",
    "Roaming charges applied despite being within home {area} circle",
]
areas = ["Sonipat", "Gurugram", "Panipat", "Rohtak", "Karnal", "Delhi NCR"]
issues = ["video streaming", "UPI payment", "OTP delivery", "customer app login"]

rows = []
for i in range(N):
    cust_id = f"TS{1000+i}"
    plan = random.choice(plan_types_clean)
    tenure = int(np.clip(np.random.exponential(scale=18), 1, 96))
    if plan == "Prepaid":
        bill = round(np.random.normal(249, 60), 2)
    elif plan == "Postpaid":
        bill = round(np.random.normal(599, 120), 2)
    else:
        bill = round(np.random.normal(999, 180), 2)
    bill = max(bill, 99)

    template = random.choice(complaint_templates)
    complaint = template.format(area=random.choice(areas), plan=plan,
                                 issue=random.choice(issues))

    resolved = random.random() < 0.72
    days_to_resolve = int(np.clip(np.random.exponential(scale=2.5), 0, 14)) if resolved else np.nan
    satisfaction = None
    if resolved:
        # faster resolution -> higher satisfaction, with noise (a Reason claim to VERIFY, not assume)
        base = 5 - (days_to_resolve / 4)
        satisfaction = int(np.clip(round(base + np.random.normal(0, 0.6)), 1, 5))

    signup_month_offset = tenure
    signup_date = pd.Timestamp("2026-07-06") - pd.DateOffset(months=int(signup_month_offset))

    rows.append({
        "customer_id": cust_id,
        "plan_type": plan,
        "monthly_bill_inr": bill,
        "tenure_months": tenure,
        "signup_date": signup_date.strftime("%Y-%m-%d"),
        "complaint_text": complaint,
        "resolved": resolved,
        "days_to_resolve": days_to_resolve,
        "satisfaction_score": satisfaction,
    })

df_raw = pd.DataFrame(rows)

# inject realistic data-quality issues (targets for Day 193's cleaning layer, not today)
dirty_idx = np.random.choice(df_raw.index, size=14, replace=False)
for j, idx in enumerate(dirty_idx):
    df_raw.loc[idx, "plan_type"] = plan_types_dirty[j % len(plan_types_dirty)]

dup_rows = df_raw.sample(n=9, random_state=SEED)
df_raw = pd.concat([df_raw, dup_rows], ignore_index=True)

print(f"TeleServe India raw dataset generated: {len(df_raw)} rows, {df_raw.shape[1]} columns")
df_raw.head()


TeleServe India raw dataset generated: 409 rows, 9 columns


,customer_id,plan_type,monthly_bill_inr,tenure_months,signup_date,complaint_text,resolved,days_to_resolve,satisfaction_score
0,TS1000,Postpaid,469.76,25,2024-06-06,Customer care call disconnected three times wi...,True,6.0,2.0
1,TS1001,Postpaid,376.93,1,2026-06-06,Data balance deducted incorrectly after using ...,True,0.0,5.0
2,TS1002,Postpaid,653.68,19,2024-12-06,Internet speed far below promised Postpaid pla...,True,3.0,4.0
3,TS1003,Prepaid,205.16,21,2024-10-06,Customer care call disconnected three times wi...,False,NaN,NaN
4,TS1004,Postpaid,495.01,62,2021-05-06,Data balance deducted incorrectly after using ...,True,3.0,4.0


---
## Section 2 — Practice Tasks

### Task 1 — Data Profile (10 pts)
Before deciding architecture, profile the raw data. Print every number below from actual output —
these numbers become the **Number** in Task 3's NRA business case later.


In [2]:
# TASK 1: Data Profile
# Goal: extract actual values from df_raw to inform architecture decisions.
# Method: pandas aggregations; all numbers are read from the data, not estimated.

total_rows = len(df_raw)                       # 409 rows (400 + 9 duplicates)
duplicate_rows = df_raw.duplicated().sum()     # 9 duplicate rows
missing_satisfaction = df_raw["satisfaction_score"].isna().sum()   # number of unresolved complaints
pct_resolved = round((df_raw["resolved"] == True).mean() * 100, 2)   # ~72%
plan_type_unique_values = df_raw["plan_type"].unique().tolist()   # reveals inconsistent casing

print(f"Total rows: {total_rows}")
print(f"Duplicate rows: {duplicate_rows}")
print(f"Missing satisfaction_score: {missing_satisfaction}")
print(f"% resolved: {pct_resolved}")
print(f"Unique plan_type values (raw): {plan_type_unique_values}")

Total rows: 409
Duplicate rows: 9
Missing satisfaction_score: 111
% resolved: 72.86
Unique plan_type values (raw): ['Postpaid', 'Prepaid', 'Broadband', 'Broadband ', 'POSTPAID', 'prepaid']


### Task 2 — Capstone Architecture Definition (25 pts)
Define the 5-layer architecture as a Python dictionary. For each layer, name the **specific
technology from what you've already learned in Months 6–11** — not a new tool, not a guess.
This is scored against a fixed answer key that maps each layer to a Month.


In [9]:
# TASK 2: Architecture Definition (CORRECTED)
# Goal: specify exact tech for each layer, including Month 9 NLP.
# Method: use learned tools from Months 6–11.

capstone_architecture = {
    "ingestion": {
        "technology": "pandas.read_csv() + boto3 (S3) if needed  (Month 11)",
        "purpose": "Load the raw TeleServe India CSV into a DataFrame, handling file paths and optional cloud storage.",
    },
    "transformation_cleaning": {
        "technology": "pandas cleaning pipeline with custom classes (Month 11)",
        "purpose": "Normalize plan_type casing, drop duplicate rows, and handle missing satisfaction scores (keep NaN for unresolved cases).",
    },
    "ml_feature_layer": {
        "technology": "scikit-learn (RandomForestClassifier) + TfidfVectorizer for complaint_text (Month 9 NLP)  (Months 6-7, 9)",
        "purpose": "Train a classifier to predict whether a complaint will be resolved, using billing/tenure features and TF‑IDF vectors from complaint texts.",
    },
    "rag_llm_layer": {
        "technology": "LangChain + FAISS vector store + Groq LLM (Month 10)",
        "purpose": "Answer natural‑language questions about complaint resolution and customer feedback by retrieving relevant complaint_text snippets and generating answers.",
    },
    "serving_monitoring": {
        "technology": "Streamlit app (Month 8) + pytest test gate (Month 11)",
        "purpose": "Provide a public UI for both ML predictions and RAG queries, with automated tests ensuring pipeline integrity.",
    },
}

for layer, details in capstone_architecture.items():
    print(f"{layer}: {details}")

ingestion: {'technology': 'pandas.read_csv() + boto3 (S3) if needed  (Month 11)', 'purpose': 'Load the raw TeleServe India CSV into a DataFrame, handling file paths and optional cloud storage.'}
transformation_cleaning: {'technology': 'pandas cleaning pipeline with custom classes (Month 11)', 'purpose': 'Normalize plan_type casing, drop duplicate rows, and handle missing satisfaction scores (keep NaN for unresolved cases).'}
ml_feature_layer: {'technology': 'scikit-learn (RandomForestClassifier) + TfidfVectorizer for complaint_text (Month 9 NLP)  (Months 6-7, 9)', 'purpose': 'Train a classifier to predict whether a complaint will be resolved, using billing/tenure features and TF‑IDF vectors from complaint texts.'}
rag_llm_layer: {'technology': 'LangChain + FAISS vector store + Groq LLM (Month 10)', 'purpose': 'Answer natural‑language questions about complaint resolution and customer feedback by retrieving relevant complaint_text snippets and generating answers.'}
serving_monitoring: {'

### Task 3 — NRA Capstone Business Case (20 pts)
Write the project charter as Number → Reason → Action, using the **actual printed values from
Task 1** — not memory, not last month's LogiTrack numbers.

- **Number:** the specific profiling stat that justifies why this dataset needs *both* an ML layer
  and an RAG layer (not just one).
- **Reason:** the causal mechanism only — stop before restating the number or jumping to a solution.
- **Action:** name the specific layer/technology decision this leads to, with no hedging.


In [10]:
# TASK 3: NRA Capstone Business Case 
# Goal: justify both ML and RAG layers using numbers from Task 1.
# Method: keep Reason as pure causal mechanism; Action as the build decision.

missing_sat = df_raw["satisfaction_score"].isna().sum()
total = len(df_raw)
unique_texts = df_raw["complaint_text"].nunique()

nra_bullet = {
    "number": f"{missing_sat} missing satisfaction scores (out of {total}) and {unique_texts} unique complaint texts",
    "reason": "Satisfaction scores are missing only for unresolved complaints, so resolution must be predicted from structured fields alone, independent of text.",
    "action": "Build a scikit‑learn classifier on structured features plus TF‑IDF from complaint texts, and a separate LangChain RAG pipeline for free‑text question answering; serve both through a single Streamlit interface.",
}

for k, v in nra_bullet.items():
    print(f"{k.upper()}: {v}")

NUMBER: 111 missing satisfaction scores (out of 409) and 35 unique complaint texts
REASON: Satisfaction scores are missing only for unresolved complaints, so resolution must be predicted from structured fields alone, independent of text.
ACTION: Build a scikit‑learn classifier on structured features plus TF‑IDF from complaint texts, and a separate LangChain RAG pipeline for free‑text question answering; serve both through a single Streamlit interface.


### Task 4 — GitHub Repo Skeleton (20 pts)
Define the folder structure for `Month12-Capstone-Netherlands-Portfolio`, mirroring the convention
already used in `Month11-DataEngineering-Portfolio` (branch `master`, PAT auth, `auto_sync` watcher).


In [11]:
# TASK 4: GitHub Repo Skeleton (CORRECTED)
# Goal: define the repository structure, including tracked deliverables.

repo_skeleton = {
    "repo_name": "Month12-Capstone-Netherlands-Portfolio",
    "branch": "master",
    "folders": [
        "notebooks/",
        "src/",
        "data_raw/",
        "data_processed/",
        ".github/workflows/",
        "docs/",
        "app/",
        "tests/",
    ],
    "key_files": [
        "README.md",
        "pipeline.py",
        "model.py",
        "rag.py",
        "app.py",
        "test_gate.py",
        "auto_sync_month12.py",
        "requirements.txt",
        "config.yaml",
        "NRA_Capstone_Business_Case.md",  
        "architecture_diagram.png",        
    ],
}

print(repo_skeleton)

{'repo_name': 'Month12-Capstone-Netherlands-Portfolio', 'branch': 'master', 'folders': ['notebooks/', 'src/', 'data_raw/', 'data_processed/', '.github/workflows/', 'docs/', 'app/', 'tests/'], 'key_files': ['README.md', 'pipeline.py', 'model.py', 'rag.py', 'app.py', 'test_gate.py', 'auto_sync_month12.py', 'requirements.txt', 'config.yaml', 'NRA_Capstone_Business_Case.md', 'architecture_diagram.png']}


### Task 5 — Success Metrics & Exit-Criteria Mapping (15 pts)
Map each of the 5 architecture layers to one measurable "done" condition, and tie the whole capstone
back to the program's stated Month 12 exit criteria (3 GitHub repos with 10+ stars, 1 deployed public
app, IELTS score, motivation letter draft, both scholarships submitted — deployed-app criterion is
what THIS capstone satisfies).


In [12]:
# TASK 5: Success Metrics 
# Goal: define one measurable condition per layer, consistent with MNAR reasoning.

success_metrics = {
    "ingestion": "Loads all 409 raw rows with 0 read errors (verified by total_rows == 409).",
    "transformation_cleaning": "plan_type casing normalized to ['Prepaid','Postpaid','Broadband']; duplicate rows reduced to 0; missing satisfaction left as NaN (no imputation).",
    "ml_feature_layer": "Model achieves F1‑score ≥ 0.75 on held‑out test set (20% split), using TF‑IDF features from complaint_text.",
    "rag_llm_layer": "RAG answers correctly on at least 8 out of 10 held‑out complaint questions (human‑evaluated).",
    "serving_monitoring": "Streamlit app is deployed to a public URL and passes `test_gate.py` with all assertions green.",
}

for k, v in success_metrics.items():
    print(f"{k}: {v}")

ingestion: Loads all 409 raw rows with 0 read errors (verified by total_rows == 409).
transformation_cleaning: plan_type casing normalized to ['Prepaid','Postpaid','Broadband']; duplicate rows reduced to 0; missing satisfaction left as NaN (no imputation).
ml_feature_layer: Model achieves F1‑score ≥ 0.75 on held‑out test set (20% split), using TF‑IDF features from complaint_text.
rag_llm_layer: RAG answers correctly on at least 8 out of 10 held‑out complaint questions (human‑evaluated).
serving_monitoring: Streamlit app is deployed to a public URL and passes `test_gate.py` with all assertions green.


### Bonus (10★) — Text Architecture Diagram
Write a simple text/ASCII or Mermaid-syntax diagram showing how the 5 layers connect end to end
(data flow direction, not just a list). This becomes `architecture_diagram` for the Day 196 README.


In [7]:
# BONUS: Text/Mermaid architecture diagram
# Goal: illustrate the end‑to‑end data flow from raw data to client output.
# Method: draw a clear directional diagram with branches for ML and RAG.

architecture_diagram = """
                   ┌─────────────────────────────────────────────┐
                   │             INGESTION                       │
                   │   (pandas.read_csv / S3)                    │
                   └─────────────────┬───────────────────────────┘
                                     │
                                     ▼
                   ┌─────────────────────────────────────────────┐
                   │         TRANSFORMATION / CLEANING          │
                   │  - Normalize plan_type                      │
                   │  - Drop duplicates                          │
                   │  - Handle missing satisfaction             │
                   └─────────────────┬───────────────────────────┘
                                     │
                   ┌─────────────────┼─────────────────────────────┐
                   │                 │                             │
                   ▼                 ▼                             ▼
    ┌─────────────────────────┐   ┌─────────────────────────────┐
    │   ML / FEATURE LAYER    │   │     RAG / LLM LAYER        │
    │  - scikit-learn         │   │  - LangChain + FAISS       │
    │  - Train classifier     │   │  - Embed complaint_text    │
    │  - Predict resolution   │   │  - Retrieve & answer Qs    │
    └────────────┬────────────┘   └──────────────┬──────────────┘
                 │                               │
                 └───────────────┬───────────────┘
                                 │
                                 ▼
                   ┌─────────────────────────────────────────────┐
                   │        SERVING / MONITORING                │
                   │  - Streamlit app (UI)                       │
                   │  - test_gate.py validates pipeline         │
                   │  - Public URL + monitoring logs            │
                   └─────────────────────────────────────────────┘
                                 │
                                 ▼
                               CLIENT
"""

print(architecture_diagram)


                   ┌─────────────────────────────────────────────┐
                   │             INGESTION                       │
                   │   (pandas.read_csv / S3)                    │
                   └─────────────────┬───────────────────────────┘
                                     │
                                     ▼
                   ┌─────────────────────────────────────────────┐
                   │         TRANSFORMATION / CLEANING          │
                   │  - Normalize plan_type                      │
                   │  - Drop duplicates                          │
                   │  - Handle missing satisfaction             │
                   └─────────────────┬───────────────────────────┘
                                     │
                   ┌─────────────────┼─────────────────────────────┐
                   │                 │                             │
                   ▼                 ▼                             ▼
    ┌──

---
## Section 3 — Scoring Rubric

| Task | Points | Criteria |
|---|---|---|
| Task 1 — Data Profile | 10 | All 5 values printed from actual code output, not estimated |
| Task 2 — Architecture Definition | 25 | 5/5 layers filled; technology must trace to a Month 6-11 skill; purpose is specific to TeleServe India, not generic |
| Task 3 — NRA Business Case | 20 | Number from Task 1 output; Reason is a causal mechanism (not an outcome restatement); Action names a specific layer/technique with no hedging |
| Task 4 — Repo Skeleton | 20 | Branch = master; folder/file list functionally matches Month 11 convention |
| Task 5 — Success Metrics | 15 | Each of 5 layers has one measurable (not vague) condition |
| **Total** | **90** | |
| Bonus — Architecture Diagram | +10★ | Diagram shows directional data flow across all 5 layers, not just a list |

**Common failure mode to self-check before submitting:** in Task 3, Reason drifting into
"...so satisfaction is lower for slow resolutions" (an outcome restatement) instead of stating
*why* the missingness exists structurally. This has been the dominant correction pattern across
Months 6-11 — don't let it carry into Month 12.


---
## Interview Framing

*"Walk me through how you scoped your final capstone project."*

> "Before writing any code, I mapped the project to five layers - ingestion, cleaning, an ML layer,
> a RAG/LLM layer, and a serving layer - and for each one I named the exact technology I'd already
> validated in an earlier part of my learning, rather than reaching for something new. I also wrote
> the business case and success metrics before touching the pipeline, so 'done' was defined up front
> instead of decided arbitrarily in the last week."

---

**Next: Day 193 — Capstone Part 2: Data Layer.** Build the actual cleaning pipeline against the
data-quality issues profiled today (6→3 plan_type values, 9 duplicates, structurally missing
satisfaction_score), with an assertion test suite gating anything downstream — same pattern as
Month 11 Day 189, one layer of a five-layer system instead of the whole thing.
